In [2]:
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/iccad-hotspot-detection
!git clone https://github.com/Jyoti-a1230/iccad-hotspot-detection.git /content/iccad-hotspot-detection

!mkdir -p /content/raw
!unzip -q /content/drive/MyDrive/iccad_raw/iccad.zip -d /content/raw

import sys
sys.path.append('/content/iccad-hotspot-detection/src')
from data import ROOT, BENCHMARKS, label_from_filename, load_clip
from model import build_model
from pipeline import build_dataset

import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image
import matplotlib.pyplot as plt

Mounted at /content/drive
Cloning into '/content/iccad-hotspot-detection'...
remote: Enumerating objects: 73, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 73 (delta 18), reused 61 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (73/73), 15.17 MiB | 23.83 MiB/s, done.
Resolving deltas: 100% (18/18), done.


In [2]:
!ls /content/drive/MyDrive/iccad_checkpoints | grep iccad2_focal

iccad2_focal_g2a75_01.keras
iccad2_focal_g2a75_04.keras
iccad2_focal_g2a75_05.keras
iccad2_focal_g2a75_06.keras
iccad2_focal_g2a75_07.keras
iccad2_focal_g2a75_12.keras
iccad2_focal_g2a75_21.keras
iccad2_focal_g2a75_28.keras
iccad2_focal_g2a75_31.keras


In [3]:
from losses import binary_focal_loss
from train import run_training
from evaluate import evaluate_on_test

In [5]:
model_focal = tf.keras.models.load_model(
    '/content/drive/MyDrive/iccad_checkpoints/iccad2_focal_g2a75_31.keras',
    compile=False
)
dummy = tf.zeros((1, 224, 224, 1))
_ = model_focal(dummy)
model_focal.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 222, 222, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 222, 222, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 109, 109, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 109, 109, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 52, 52, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 52, 52, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 118,401 (462.50 KB)

 Trainable params: 117,953 (460.75 KB)

 Non-trainable params: 448 (1.75 KB)

In [ ]:
from gradcam import build_grad_model,show_gradcam,load_for_gradcam,
from evaluate import build_test_df,build_dataset

In [6]:
grad_model_focal = build_grad_model(model_focal, "PASTE_LAYER_NAME")

test_df = build_test_df("iccad2", ROOT).sort_values("filename").reset_index(drop=True)
test_ds = build_dataset(test_df, benchmark="iccad2", split="test", root=ROOT, shuffle=False)
test_preds_focal = model_focal.predict(test_ds)

NameError: name 'build_grad_model' is not defined